# Week 3: Model Implementation — Term Deposit Subscription Prediction
**Bank Marketing Dataset | Machine Learning Engineer Internship**

This notebook implements a baseline Logistic Regression model to predict whether a bank client
will subscribe to a term deposit, following the preprocessing strategy defined in Week 2.

**Sections:**
1. Load Data
2. Preprocessing (per Week 2 plan)
3. Train—Test Split
4. Model Implementation: Logistic Regression
5. Practical Demonstration on Sample Records
6. Notes on Workflow and Debugging

## 1. Load Data

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("bank-additional.csv", sep=";")
print("Shape:", df.shape)
df.head()

Shape: (4119, 21)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,30,blue-collar,married,basic.9y,no,yes,no,cellular,may,fri,...,2,999,0,nonexistent,-1.8,92.893,-46.2,1.313,5099.1,no
1,39,services,single,high.school,no,no,no,telephone,may,fri,...,4,999,0,nonexistent,1.1,93.994,-36.4,4.855,5191.0,no
2,25,services,married,high.school,no,yes,no,telephone,jun,wed,...,1,999,0,nonexistent,1.4,94.465,-41.8,4.962,5228.1,no
3,38,services,married,basic.9y,no,unknown,unknown,telephone,jun,fri,...,3,999,0,nonexistent,1.4,94.465,-41.8,4.959,5228.1,no
4,47,admin.,married,university.degree,no,yes,no,cellular,nov,mon,...,1,999,0,nonexistent,-0.1,93.200,-42.0,4.191,5195.8,no


## 2. Preprocessing (per Week 2 Strategy)

Recap of decisions made in Week 2, now implemented:
- `pdays` sentinel (999) split into `contacted_before` flag + cleaned numeric column
- `duration` dropped (data leakage risk — not known before a call happens)
- Nominal categoricals one-hot encoded; `education` ordinal encoded; binary yes/no/unknown fields mapped
- Numeric features standardized

In [2]:
# Resolve the pdays sentinel value
df['contacted_before'] = (df['pdays'] != 999).astype(int)
df['pdays_clean'] = df['pdays'].replace(999, 0)
df = df.drop(columns=['pdays'])

# Drop duration to avoid data leakage (see Week 1/2 discussion)
df_deploy = df.drop(columns=['duration'])

print("Columns after cleaning:", df_deploy.shape[1])

Columns after cleaning: 21


In [3]:
# Encode categorical variables
nominal_cols = ['job', 'marital', 'contact', 'poutcome', 'month', 'day_of_week']
df_encoded = pd.get_dummies(df_deploy, columns=nominal_cols)

# Ordinal encode education (has a genuine natural order)
edu_order = ['unknown', 'illiterate', 'basic.4y', 'basic.6y', 'basic.9y',
             'high.school', 'professional.course', 'university.degree']
df_encoded['education'] = df_deploy['education'].map({v: i for i, v in enumerate(edu_order)})

# Map binary yes/no/unknown fields; unknown kept as its own signal (-1)
for col in ['default', 'housing', 'loan']:
    df_encoded[col] = df_deploy[col].map({'no': 0, 'yes': 1, 'unknown': -1})

# Encode target
df_encoded['y'] = df_encoded['y'].map({'no': 0, 'yes': 1})

print("Final feature matrix shape:", df_encoded.shape)

Final feature matrix shape: (4119, 51)


In [4]:
from sklearn.preprocessing import StandardScaler

numeric_cols = ['age', 'campaign', 'previous', 'pdays_clean',
                'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']

scaler = StandardScaler()
df_encoded[numeric_cols] = scaler.fit_transform(df_encoded[numeric_cols])

df_encoded[numeric_cols].describe().round(2)

,age,campaign,previous,pdays_clean,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
count,4119.00,4119.00,4119.00,4119.00,4119.00,4119.00,4119.00,4119.00,4119.00
mean,-0.00,0.00,-0.00,0.00,-0.00,-0.00,-0.00,0.00,0.00
std,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
min,-2.14,-0.60,-0.35,-0.17,-2.23,-2.38,-2.24,-1.72,-2.75
25%,-0.79,-0.60,-0.35,-0.17,-1.21,-0.87,-0.48,-1.32,-0.91
50%,-0.20,-0.21,-0.35,-0.17,0.65,0.29,-0.28,0.71,0.33
75%,0.67,0.18,-0.35,-0.17,0.84,0.72,0.89,0.77,0.84
max,4.64,12.64,10.72,15.17,0.84,2.05,2.96,0.82,0.84


## 3. Train–Test Split

Using a stratified split so both sets preserve the original ~89/11 class ratio.

In [5]:
from sklearn.model_selection import train_test_split

X = df_encoded.drop(columns=['y'])
y = df_encoded['y']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train class balance:\n", y_train.value_counts(normalize=True).round(3))
print("Test class balance:\n", y_test.value_counts(normalize=True).round(3))

Train shape: (3295, 50)
Test shape: (824, 50)
Train class balance:
 y
0    0.89
1    0.11
Name: proportion, dtype: float64
Test class balance:
 y
0    0.891
1    0.109
Name: proportion, dtype: float64


## 4. Model Implementation: Logistic Regression

**Why Logistic Regression as the baseline:**
- Interpretable — coefficients directly show how each feature pushes the prediction toward
  "yes" or "no", which matters for a bank deciding who to prioritize calling.
- Fast to train and a sensible baseline before trying more complex models later.

**Hypothesis:** client demographic, financial, and campaign-contact features contain enough
signal to meaningfully separate clients likely to subscribe from those who aren't — better
than random guessing or a majority-class baseline.

**Assumptions:** Logistic Regression assumes a roughly linear relationship between the
features and the log-odds of the outcome. This is a simplifying assumption — some real
relationships (e.g. age) may be non-linear — but it's a reasonable, interpretable starting point.

**Handling class imbalance:** `class_weight='balanced'` is used so the model doesn't just learn
to predict "no" for everyone, which would otherwise score deceptively high on raw accuracy.

In [6]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
model.fit(X_train, y_train)

print("Model trained successfully.")
print("Number of features:", X_train.shape[1])

Model trained successfully.
Number of features: 50


### Preliminary Performance Check

A full evaluation (metrics, cross-validation, error analysis) is the dedicated focus of Week 4 — this is a quick sanity check to confirm the model has actually learned something before moving on.

In [7]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

y_pred = model.predict(X_test)

print("Accuracy: ", round(accuracy_score(y_test, y_pred), 3))
print("Precision:", round(precision_score(y_test, y_pred), 3))
print("Recall:   ", round(recall_score(y_test, y_pred), 3))
print("F1 Score: ", round(f1_score(y_test, y_pred), 3))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:  0.846
Precision: 0.372
Recall:    0.6
F1 Score:  0.46

Confusion Matrix:
[[643  91]
 [ 36  54]]


**Reading these numbers honestly:** Accuracy alone (~85%) would look good but is misleading
given the 89/11 class imbalance — a model that always predicts "no" would score ~89% accuracy
while being completely useless. Recall of ~60% is the more meaningful number here: it means the
model correctly identifies roughly 6 out of 10 clients who would actually subscribe, which is
far better than the ~11% a random guess would catch. Precision is lower, meaning there are false
positives — clients flagged as likely subscribers who say no. This precision/recall trade-off,
and whether to shift it, is exactly what Week 4 and Week 5 will dig into properly.

## 5. Practical Demonstration on Sample Records

Testing the trained model on a few individual records from the test set, to confirm it produces
sensible, well-formed predictions on real (unseen) data — not just aggregate metrics.

In [8]:
# Take 5 sample records from the test set
sample = X_test.iloc[:5]
sample_actual = y_test.iloc[:5]

sample_pred = model.predict(sample)
sample_proba = model.predict_proba(sample)[:, 1]

results = pd.DataFrame({
    'actual': sample_actual.values,
    'predicted': sample_pred,
    'predicted_probability_yes': sample_proba.round(3)
})
results

,actual,predicted,predicted_probability_yes
0,0,0,0.447
1,0,0,0.439
2,0,0,0.129
3,0,0,0.261
4,0,0,0.452


**Error handling consideration:** in a real deployment, this prediction step would need to
guard against malformed input — e.g. a new client record missing a column the model expects,
or a categorical value unseen during training (like a new job category). A production version
of this pipeline would wrap `model.predict()` in a try/except block and validate the incoming
record's columns against `X_train.columns` before prediction, raising a clear error rather than
failing silently or crashing.

In [9]:
# Example of a defensive prediction function with basic error handling
def safe_predict(record_df, model, expected_columns):
    missing = set(expected_columns) - set(record_df.columns)
    if missing:
        raise ValueError(f"Missing expected columns: {missing}")
    record_df = record_df[expected_columns]  # enforce correct column order
    try:
        return model.predict(record_df), model.predict_proba(record_df)[:, 1]
    except Exception as e:
        raise RuntimeError(f"Prediction failed: {e}")

pred, proba = safe_predict(X_test.iloc[:3], model, X_train.columns)
print("Predictions:", pred)
print("Probabilities:", proba.round(3))

Predictions: [0 0 0]
Probabilities: [0.447 0.439 0.129]


## 6. Notes on Workflow, Versioning, and Testing

- **Code organization:** preprocessing steps are kept in the same order as the Week 2 strategy
  document, so each transformation can be traced back to the finding that motivated it.
- **Experiment tracking:** this baseline run (Logistic Regression, `class_weight='balanced'`,
  `random_state=42`) is the reference point for Week 5's tuning experiments — any future model
  will be compared against these exact numbers, using the same train/test split (fixed random
  seed) for a fair comparison.
- **Version control:** this notebook is committed to the project's GitHub repository alongside
  the Week 1 and Week 2 planning documents, so the full history of decisions is visible.
- **Verifying functionality:** the notebook was run top-to-bottom with a fresh kernel to confirm
  every cell executes without errors before submission — the same check performed for this
  submitted version.